# Load Regression XGBoost & Comparison Plots

Drop-in replacement for the Lasso notebook. Same data loading, same LeaveOneGroupOut-by-year CV.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import root_mean_squared_error

import xgboost as xgb
from pygam import LinearGAM, s, f, te

import os


In [3]:
import xarray as xr
ds = xr.open_dataset(r"/gdex/data/d651009/b.e13.BRCP85C5.ne120_t12.cesm-ihesp-hires1.0.30-2006-2100.002/atm/proc/tseries/hour_6/b.e13.BRCP85C5.ne120_t12.cesm-ihesp-hires1.0.30-2006-2100.002.cam.h2.U10.2015010100-2016010100.nc")
print(ds["time_bnds"].isel(time=0).values)

ds

[cftime.DatetimeNoLeap(2014, 12, 31, 18, 0, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2015, 1, 1, 0, 0, 0, 0, has_year_zero=True)]


<xarray.Dataset> Size: 5GB
Dimensions:        (lev: 30, ilev: 31, cosp_prs: 7, cosp_tau: 7, cosp_scol: 50,
                    cosp_ht: 40, cosp_sr: 15, cosp_sza: 5, nbnd: 2,
                    ncol: 777602, time: 1460, chars: 8)
Coordinates:
  * lev            (lev) float64 240B 3.643 7.595 14.36 ... 957.5 976.3 992.6
  * ilev           (ilev) float64 248B 2.255 5.032 10.16 ... 967.5 985.1 1e+03
  * cosp_prs       (cosp_prs) float64 56B 900.0 740.0 620.0 ... 375.0 245.0 90.0
  * cosp_tau       (cosp_tau) float64 56B 0.15 0.8 2.45 6.5 16.2 41.5 219.5
  * cosp_scol      (cosp_scol) int32 200B 1 2 3 4 5 6 7 ... 44 45 46 47 48 49 50
  * cosp_ht        (cosp_ht) float64 320B 240.0 720.0 ... 1.848e+04 1.896e+04
  * cosp_sr        (cosp_sr) float64 120B 0.605 2.1 4.0 ... 70.0 539.5 1.004e+03
  * cosp_sza       (cosp_sza) float64 40B 0.0 15.0 30.0 45.0 60.0
  * time           (time) object 12kB 2015-01-01 00:00:00 ... 2015-12-31 18:0...
Dimensions without coordinates: nbnd, ncol, chars
Data variables: (12/35)
    hyam           (lev) float64 240B ...
    hybm           (lev) float64 240B ...
    P0             float64 8B ...
    hyai           (ilev) float64 248B ...
    hybi           (ilev) float64 248B ...
    cosp_prs_bnds  (cosp_prs, nbnd) float64 112B ...
    ...             ...
    n2ovmr         (time) float64 12kB ...
    f11vmr         (time) float64 12kB ...
    f12vmr         (time) float64 12kB ...
    sol_tsi        (time) float64 12kB ...
    nsteph         (time) int32 6kB ...
    U10            (time, ncol) float32 5GB ...
Attributes:
    np:               4
    ne:               120
    Conventions:      CF-1.0
    source:           CAM
    case:             b.e13.BRCP85C5.ne120_t12.cesm-ihesp-hires1.0.30.002
    title:            UNSET
    logname:          nanr
    host:             login1.frontera.
    Version:          $Name$
    revision_Id:      $Id$
    initial_file:     b.e13.BHISTC5.ne120_t12.cesm-ihesp-hires1.0.30-1920-210...
    topography_file:  /work/02503/edwardsj/CESM/inputdata//atm/cam/topo/USGS-...

## Data Loading
Identical to the Lasso notebook.

In [ ]:
years = [2014, 2015, 2016, 2017, 2018, 2019, 2021, 2022, 2023, 2024]

load_data = pd.DataFrame(columns=["LoadResource Zone", "ActualLoad (MWh)"])
for year in years:
    load_ts = pd.read_excel(
        rf"./MISO historical loads/{year}1231_dfal_HIST.xls",
        index_col=0, header=5
    )
    load_ts = load_ts[:-2].drop(columns=["MTLF (MWh)"])
    load_ts = load_ts[~(load_ts.index == "MarketDay")]
    load_ts.index = pd.to_datetime(load_ts.index)
    load_ts.index = load_ts.index + pd.to_timedelta(load_ts["HourEnding"] - 1, unit="h")
    load_data = pd.concat([load_data, load_ts[["LoadResource Zone", "ActualLoad (MWh)"]]], axis=0)

temp_data = pd.DataFrame(columns=["MISO-0001","MISO-0027","MISO-0035","MISO-0004","MISO-0006","MISO-8910"])
for year in years:
    temp_ts = pd.read_csv(rf"./ERA5/MISO_t2m_{year}.csv", index_col=0, header=0, parse_dates=True)
    temp_data = pd.concat([temp_data, temp_ts], axis=0)

lrz_ids = ["MISO-0001","MISO-0027","MISO-0035","MISO-0004","MISO-0006","MISO-8910"]
load_wide = load_data.pivot_table(index=load_data.index, columns="LoadResource Zone", values="ActualLoad (MWh)")
load_wide["LRZ8_9_10"] = load_wide["LRZ8_9_10"].fillna(0) + load_wide["LRZ8_9"].fillna(0)
load_wide = load_wide.drop(columns=["LRZ8_9", "MISO"])
load_wide = load_wide.rename(columns=dict(zip(load_wide.columns, lrz_ids)))

# resample to 6-hourly
temp_data = temp_data.tz_localize("UTC").resample("6h").mean()
load_wide.index = pd.to_datetime(load_wide.index).tz_localize("Etc/GMT+5").tz_convert("UTC")
load_wide = load_wide.resample("6h").mean()

# convert to DST
temp_data.index = temp_data.index.tz_convert("America/Indiana/Indianapolis")
load_wide.index  = load_wide.index.tz_convert("America/Indiana/Indianapolis")

common_index = temp_data.index.intersection(load_wide.index)
temp_data = temp_data.loc[common_index]
load_wide = load_wide.loc[common_index]

valid_rows = temp_data.notna().all(axis=1) & load_wide.notna().all(axis=1)
temp_data = temp_data.loc[valid_rows]
load_data = load_wide.loc[valid_rows]
print(f"Data shape: temp={temp_data.shape}, load={load_data.shape}")


## Feature Engineering

Raw numeric features no dummies needed for tree models; GAM applies splines internally.

In [ ]:
HOLIDAYS       = ["memorial_day", "july_4th", "labor_day", "christmas", "new_years"]
HOLIDAY_WINDOW = 2  # days on each side

def get_holiday_dates(years):
    """Return {holiday: frozenset of tz-naive midnight Timestamps} for years-1 through years+1."""
    year_range = range(min(years) - 1, max(years) + 2)
    dates = {h: set() for h in HOLIDAYS}
    for year in year_range:
        may_31 = pd.Timestamp(year, 5, 31)
        dates["memorial_day"].add(may_31 - pd.Timedelta(days=may_31.dayofweek))  # last Mon of May
        dates["july_4th"].add(pd.Timestamp(year, 7, 4))
        sep_1 = pd.Timestamp(year, 9, 1)
        dates["labor_day"].add(sep_1 + pd.Timedelta(days=(7 - sep_1.dayofweek) % 7))  # first Mon of Sep
        dates["christmas"].add(pd.Timestamp(year, 12, 25))
        dates["new_years"].add(pd.Timestamp(year, 1, 1))
    return {h: frozenset(v) for h, v in dates.items()}


def build_features_raw(temp, lrz):
    t = temp[lrz]
    t_f = t * 9 / 5 + 32
    X = pd.DataFrame(index=t.index)

    X["temp"]         = t
    # X["temp_squared"] = t ** 2
    X["temp_lag1"]    = t.shift(1)
    # X["temp_lag2"]    = t.shift(2)
    # X["temp_lag3"]    = t.shift(3)
    # X["temp_lag4"]    = t.shift(4)
    X["temp_lag24"]   = t.shift(24)

    # 10°F daily-average temp bin (broadcast to all hours of that day)
    daily_avg_f = t_f.resample("D").mean().reindex(t.index, method="ffill")
    bin_edges = np.arange(
        np.floor(daily_avg_f.min() / 10) * 10,
        np.ceil(daily_avg_f.max() / 10) * 10 + 10,
        10
    )
    # X["temp_bin"] = pd.cut(daily_avg_f, bins=bin_edges, labels=False).astype(float)

    X["hour"]        = t.index.hour
    X["day_of_week"] = t.index.day_of_week
    # X["weekend"]     = (t.index.day_of_week > 4).astype(int)
    X["month"]       = t.index.month
    X["year"]        = t.index.year

    # Holiday window features
    # Encoding: 0 = not in window, 1 = -2 days, 2 = -1 day, 3 = day-of,
    #           4 = +1 day, 5 = +2 days
    date_index = t.index.normalize()
    if date_index.tz is not None:
        date_index = date_index.tz_localize(None)

    holiday_dates = get_holiday_dates(t.index.year.unique())
    for holiday, day_set in holiday_dates.items():
        col = np.zeros(len(t.index), dtype=int)
        for level, offset in enumerate(range(-HOLIDAY_WINDOW, HOLIDAY_WINDOW + 1), start=1):
            shifted = frozenset(d + pd.Timedelta(days=offset) for d in day_set)
            col[date_index.isin(shifted)] = level
        X[holiday] = col

    return X

# column indices for GAM:
# 0:temp  1:temp_sq  2:lag1  3:lag24  4:temp_bin  5:hour  6:dow  7:weekend  8:year
GAM_COLS = ["temp","temp_squared","temp_lag1","temp_lag24",
            "temp_bin","hour","day_of_week", "month", "year"]
# column order for GAM term indices
# 0:temp  1:temp_sq  2:lag1  3:lag24  4:hour  5:month  6:dow  7:weekend  8:year
# GAM_COLS = ["temp","temp_squared","temp_lag1","temp_lag24",
#             "hour","month","day_of_week","weekend","year"]


In [ ]:
seed_value = 42
rng = np.random.default_rng(seed_value)

train_years = rng.choice(years, 8, replace=False)
test_years =  [y for y in years if y not in train_years]

temp = temp_data[temp_data.index.year.isin(train_years)]
load = load_data[load_data.index.year.isin(train_years)]

test_years

## XGBoost

One model per CV fold (leave-one-year-out). The last training year is held out as the early-stopping validation set.

In [ ]:
# Toggle: exclude top-percentile load hours from training 
# When True, the top EXTREME_PERCENTILE% of load hours are dropped from each
# fold's training set. Predictions and evaluation always use the full test set.
EXCLUDE_EXTREMES    = True
EXTREME_PERCENTILE  = 99   # i.e. top 1%

HIGH_LOAD_WEIGHT_PERCENTILE = 90
HIGH_LOAD_WEIGHT = 1

In [ ]:
zone = "MISO-0001"  # change to any of the 6 zones

X_all = build_features_raw(temp, zone).dropna()
y_all = load[zone].loc[X_all.index]
groups = X_all.index.year

xgb_params = dict(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=10,
    reg_alpha=1.0,
    reg_lambda=5.0,
    tree_method="hist",
    early_stopping_rounds=30,
    eval_metric="rmse",
    random_state=42,
)

logo = LeaveOneGroupOut()
xgb_oof = pd.Series(index=y_all.index, dtype=float, name="xgb_pred")
xgb_records = []

for fold_idx, (train_idx, test_idx) in enumerate(logo.split(X_all, y_all, groups)):
    X_tr, X_te = X_all.iloc[train_idx], X_all.iloc[test_idx]
    y_tr, y_te = y_all.iloc[train_idx], y_all.iloc[test_idx]
    held_out_year = int(X_te.index.year[0])

    if EXCLUDE_EXTREMES:
        train_thresh = np.percentile(y_tr, EXTREME_PERCENTILE)
        keep_train = y_tr <= train_thresh
        X_tr_fit, y_tr_fit = X_tr[keep_train], y_tr[keep_train]

        test_thresh = np.percentile(y_te, 0) #EXTREME_PERCENTILE)
        keep_test = y_te > test_thresh
        test_idx_extreme = np.array(test_idx)[keep_test.values]
        X_te_eval, y_te_eval = X_te[keep_test], y_te[keep_test]
    else:
        X_tr_fit, y_tr_fit = X_tr, y_tr
        test_idx_extreme = test_idx
        X_te_eval, y_te_eval = X_te, y_te

    # Sample weights: top HIGH_LOAD_WEIGHT_PERCENTILE% of training y -> HIGH_LOAD_WEIGHT
    w_thresh = np.percentile(y_tr_fit, HIGH_LOAD_WEIGHT_PERCENTILE)
    sample_weights = np.where(y_tr_fit.values >= w_thresh, HIGH_LOAD_WEIGHT, 1.0)

    val_mask = X_tr_fit.index.year == X_tr_fit.index.year.max()
    train_weights = sample_weights[~val_mask]
    val_weights   = sample_weights[val_mask]

    model_xgb = xgb.XGBRegressor(**xgb_params)
    model_xgb.fit(
        X_tr_fit[~val_mask], y_tr_fit[~val_mask],
        sample_weight=train_weights,
        eval_set=[(X_tr_fit[val_mask], y_tr_fit[val_mask])],
        sample_weight_eval_set=[val_weights],
        verbose=False,
    )
    preds = model_xgb.predict(X_te_eval)
    xgb_oof.iloc[test_idx_extreme] = preds

    for ts, actual, predicted in zip(y_te_eval.index, y_te_eval.values, preds):
        xgb_records.append({
            "timestamp":     ts,
            "actual":        actual,
            "predicted":     predicted,
            "zone":          zone,
            "fold_idx":      fold_idx,
            "held_out_year": held_out_year,
        })

predicted_mask = xgb_oof.notna()
rmse_xgb = root_mean_squared_error(y_all[predicted_mask], xgb_oof[predicted_mask])
label = f"train≤p{EXTREME_PERCENTILE}, eval>p{EXTREME_PERCENTILE}" if EXCLUDE_EXTREMES else "all data"
print(f"XGBoost OOF RMSE ({zone}, {label}): {rmse_xgb:,.1f} MWh  [n={predicted_mask.sum()}]")

# Save to parquet (same schema as gam/lasso worker outputs)
os.makedirs(r".\xgb_results", exist_ok=True)
xgb_out = pd.DataFrame(xgb_records).set_index("timestamp")
xgb_out.to_parquet(fr".\xgb_results\xgb_preds_{zone}.parquet")
print(f"Saved → xgb_results/xgb_preds_{zone}.parquet")


In [ ]:
# feature importance (refit on all data) 
params_no_es = {k: v for k, v in xgb_params.items() if k != 'early_stopping_rounds'}
model_xgb_full = xgb.XGBRegressor(**params_no_es)
model_xgb_full.fit(X_all, y_all, verbose=False)

imp = pd.Series(model_xgb_full.feature_importances_, index=X_all.columns)
imp.sort_values().plot.barh(figsize=(7, 4), title=f"XGBoost feature importance {zone}")
plt.tight_layout(); plt.show()


In [ ]:
import shap

# ── 1. Full-data model for SHAP attribution ───────────────────────────────────
# OOF preds above remain the honest eval metric; this model is for attribution only.
# Use best_iteration from the last CV fold as a reasonable n_estimators proxy.
# Use a stratified subsample for interaction analysis
N_SHAP = 5_000
rng = np.random.default_rng(42)
shap_idx = rng.choice(len(X_all), size=min(N_SHAP, len(X_all)), replace=False)
X_shap_sample = X_all.iloc[shap_idx]

xgb_params_shap = {k: v for k, v in xgb_params.items()
                   if k not in ("early_stopping_rounds", "eval_metric")}
xgb_params_shap["n_estimators"] = model_xgb.best_iteration + 1

w_thresh_all = np.percentile(y_all, HIGH_LOAD_WEIGHT_PERCENTILE)
sw_all = np.where(y_all.values >= w_thresh_all, HIGH_LOAD_WEIGHT, 1.0)

model_shap = xgb.XGBRegressor(**xgb_params_shap)
model_shap.fit(X_all, y_all, sample_weight=sw_all, verbose=False)
print(f"Full-data model trained  (n_estimators={xgb_params_shap['n_estimators']})")

# ── 2. SHAP interaction values ────────────────────────────────────────────────
# Output shape: (n_samples, n_features, n_features)
#   shap_ia[i, j, j]  → main effect of feature j for observation i
#   shap_ia[i, j, k]  → pairwise interaction between j and k  (j ≠ k)
# Memory note: F≈20 features, N≈50k rows → ~160 MB float64 — manageable
explainer_ia = shap.TreeExplainer(model_shap)
shap_ia = explainer_ia.shap_interaction_values(X_shap_sample)  # (N, F, F)

features = X_shap_sample.columns.tolist()
n_features = len(features)
print(f"SHAP interaction tensor shape: {shap_ia.shape}  (samples × features × features)")

# ── 3. Mean |interaction| matrix (F × F) ──────────────────────────────────────
mean_abs_ia = np.abs(shap_ia).mean(axis=0)
ia_df = pd.DataFrame(mean_abs_ia, index=features, columns=features)

# ── 4. Ranked pairwise interactions (off-diagonal, symmetrized) ───────────────
upper_j, upper_k = np.triu_indices(n_features, k=1)
pair_strengths = pd.Series(
    {(features[j], features[k]): mean_abs_ia[j, k] + mean_abs_ia[k, j]
     for j, k in zip(upper_j, upper_k)},
    name="mean_abs_interaction_MWh",
).sort_values(ascending=False)

print(f"\nTop 10 pairwise SHAP interactions — {zone}:")
print(pair_strengths.head(10).to_string())


# # ── Optional: save interaction matrix ────────────────────────────────────────
# os.makedirs(r".\xgb_results", exist_ok=True)
# ia_df.to_csv(fr".\xgb_results\shap_interactions_{zone}.csv")
# print(f"Saved → xgb_results/shap_interactions_{zone}.csv")

In [ ]:

# ── 5. Interactive heatmap (Plotly) ───────────────────────────────────────────
fig = go.Figure(go.Heatmap(
    z=ia_df.values,
    x=features,
    y=features,
    colorscale="YlOrRd",
    colorbar=dict(title="Mean |SHAP interaction| (MWh)"),
    # text=np.round(ia_df.values, 1),
    # texttemplate="%{text}",
    # textfont=dict(size=10),
    hovertemplate="<b>%{y}</b> × <b>%{x}</b><br>Mean |interaction|: %{z:.1f} MWh<extra></extra>",
))

fig.update_layout(
    title=dict(
        text=f"SHAP Interaction Values — {zone}<br>"
             f"<sup>Full-data model · mean |value| across {len(X_all):,} samples</sup>",
        x=0.5,
    ),
    xaxis=dict(tickangle=-45, side="bottom"),
    yaxis=dict(autorange="reversed"),
    width=700 + n_features * 20,
    height=600 + n_features * 15,
    margin=dict(l=160, b=160, t=100, r=60),
)
fig.show()


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── SHAP Dependence Plots ─────────────────────────────────────────────────────
# Uses shap_ia (already computed): diagonal = main effect, off-diagonal = interaction
# Colored by the strongest interacting feature's value (standard dependence plot convention)

def get_strongest_interactor(feature, features, pair_strengths):
    """Return the feature with the strongest pairwise interaction."""
    candidates = {
        (a, b): v for (a, b), v in pair_strengths.items()
        if feature in (a, b)
    }
    if not candidates:
        return None
    best_pair = max(candidates, key=candidates.get)
    return best_pair[1] if best_pair[0] == feature else best_pair[0]

# Features to plot — change this list or set to `features` for all
PLOT_FEATURES = features  # or e.g. ["temp", "hour", "temp_lag1"]

n_cols = 3
n_rows = int(np.ceil(len(PLOT_FEATURES) / n_cols))

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles= [
        f"{feat}  ✕  {get_strongest_interactor(feat, features, pair_strengths) or '—'}"
        for feat in PLOT_FEATURES
    ],
    horizontal_spacing=0.08,
    vertical_spacing=0.12,
)

X_plot = X_shap_sample if "X_shap_sample" in dir() else X_all  # use sample if available

for idx, feat in enumerate(PLOT_FEATURES):
    row, col = divmod(idx, n_cols)
    fi = features.index(feat)

    y_shap = shap_ia[:, fi, fi]           # main effect
    x_feat = X_plot[feat].values

    interactor = get_strongest_interactor(feat, features, pair_strengths)
    if interactor:
        color_vals = X_plot[interactor].values
        colorbar_title = interactor
    else:
        color_vals = x_feat
        colorbar_title = feat

    scatter = go.Scatter(
        x=x_feat,
        y=y_shap,
        mode="markers",
        marker=dict(
            color=color_vals,
            colorscale="RdBu_r",
            size=3,
            opacity=0.6,
            colorbar=dict(
                title="",
                len=0.3,
                x=1.01,
            ) if idx == 0 else None,
            showscale=(idx == 0),
        ),
        hovertemplate=(
            f"<b>{feat}</b>: %{{x:.2f}}<br>"
            f"SHAP: %{{y:.1f}} MWh<br>"
            f"{interactor}: %{{marker.color:.2f}}<extra></extra>"
        ),
        showlegend=False,
    )
    fig.add_trace(scatter, row=row + 1, col=col + 1)
    fig.update_xaxes(title_text=feat, row=row + 1, col=col + 1)
    fig.update_yaxes(title_text="SHAP value (MWh)", row=row + 1, col=col + 1)

fig.update_layout(
    title=dict(
        text=f"SHAP Dependence Plots — {zone}<br>"
            f"<sup>Color = strongest interacting feature  (title format: feature ✕ color-feature)</sup>",
        x=0.5,
    ),
    height=350 * n_rows,
    width=1100,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
fig.update_xaxes(showgrid=True, gridcolor="lightgrey")
fig.update_yaxes(showgrid=True, gridcolor="lightgrey", zeroline=True, zerolinecolor="grey")
fig.show()


## Model Comparison

In [ ]:
import glob

EXCLUDE_EXTREMES = False
#  Folder selection 
subfolder = {"GAM": "extremes_weighted",
             "LASSO": "extremes_weighted",
             "XGB" : "extremes_weighted",
             "MLP" : "extremes_unweighted_v6",
             "MLP 6H" : "extremes_unweighted_v6_6H",
             "MLP test" : "",
             "ResNet" : "extremes_unweighted",}

#  Load XGB results 
xgb_parts = []
for path in glob.glob(rf"./xgb_results/{subfolder['XGB']}/xgb_preds_*.parquet"):
    xgb_parts.append(pd.read_parquet(path))
xgb_df = pd.concat(xgb_parts).sort_index() if xgb_parts else pd.DataFrame()

#  Load GAM results 
gam_parts = []
for path in glob.glob(rf"./gam_results/{subfolder['GAM']}/gam_preds_zone*_fold*.parquet"):
    gam_parts.append(pd.read_parquet(path))
gam_df = pd.concat(gam_parts).sort_index() if gam_parts else pd.DataFrame()

#  Load MLP results 
mlp_parts = []
for path in glob.glob(rf"./mlp_results/{subfolder['MLP']}/mlp_preds_zone*_fold*.parquet"):
    mlp_parts.append(pd.read_parquet(path))
mlp_df = pd.concat(mlp_parts).sort_index() if mlp_parts else pd.DataFrame()

#  Load 6 hourly MLP results 
mlpp_parts = []
for path in glob.glob(rf"./mlp_results/{subfolder['MLP 6H']}/mlp_preds_zone*_fold*.parquet"):
    mlpp_parts.append(pd.read_parquet(path))
mlpp_df = pd.concat(mlpp_parts).sort_index() if mlpp_parts else pd.DataFrame()

#  Load MLP test data results 
mlptest_parts = []
for path in glob.glob(rf"./mlp_test_results/mlp_test_preds_zone*.parquet"):
    mlptest_parts.append(pd.read_parquet(path))
mlptest_df = pd.concat(mlptest_parts).sort_index() if mlptest_parts else pd.DataFrame()

#  Load ResNet results 
resnet_parts = []
for path in glob.glob(rf"./resnet_results/{subfolder['ResNet']}/resnet_preds_zone*_fold*.parquet"):
    resnet_parts.append(pd.read_parquet(path))
resnet_df = pd.concat(resnet_parts).sort_index() if resnet_parts else pd.DataFrame()

# Load LASSO results 
lasso_parts = []
for path in glob.glob(rf"./lasso_results/{subfolder['LASSO']}/lasso_preds_zone*_fold*.parquet"):
    lasso_parts.append(pd.read_parquet(path))
lasso_df = pd.concat(lasso_parts).sort_index() if lasso_parts else pd.DataFrame()

# Plot 
LRZ_IDS = ["MISO-0001", "MISO-0027", "MISO-0035", "MISO-0004", "MISO-0006", "MISO-8910"]
n_cols = 3
n_rows = 2

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=LRZ_IDS,
    horizontal_spacing=0.08,
    vertical_spacing=0.12,
)


for i, zone in enumerate(LRZ_IDS):
    row = i // n_cols + 1
    col = i  % n_cols + 1
    axis_suffix = "" if i == 0 else str(i + 1)

    all_vals = []

    for df, label, color in [
        (lasso_df, "LASSO", "#F1D313"),
        (xgb_df, "XGBoost", "#2196F3"),
        (gam_df, "GAM",     "#F44336"),
        (mlp_df, "MLP",     "#54C82D"),
        (mlpp_df, "MLP 6H",  "#3F9724"),
        (mlptest_df, "MLP Test",  "#B076D6"),
        (resnet_df, "ResNet","#FF3CD1"),
    ]:
        if df.empty:
            continue
        z = df[df["zone"] == zone]
        if z.empty:
            continue
        
        if EXCLUDE_EXTREMES and (label != "LASSO"):
            z=z.nlargest(int(len(z) * 0.01), "actual")

        if label == "MLP-Q":
            z["predicted"] = z["q75"]

        all_vals.extend(z["actual"].tolist() + z["predicted"].tolist())
        rmse = root_mean_squared_error(z["actual"], z["predicted"])
        timestamps = z.index.strftime("%Y-%m-%d %H:%M")

        fig.add_trace(
            go.Scattergl(
                x=z["actual"],
                y=z["predicted"],
                mode="markers",
                name=label,
                legendgroup=label,
                showlegend=(i == 0),
                marker=dict(size=5, opacity=0.5, color=color),
                customdata=timestamps,
                hovertemplate=(
                    "<b>%{customdata}</b><br>"
                    "Actual: %{x:,.0f} MWh<br>"
                    "Predicted: %{y:,.0f} MWh<extra>" + label + "</extra>"
                ),
            ),
            row=row, col=col,
        )

        # fig.add_annotation(
        #     text=f"{label} RMSE: {rmse:,.0f}",
        #     xref=f"x{i+1}", yref=f"y{i+1}",
        #     x=0.03, y=0.97 - (0 if label == "XGBoost" else 0.08),
        #     xanchor="left", yanchor="top",
        #     showarrow=False,
        #     font=dict(size=10, color=color),
        #     bgcolor="rgba(255,255,255,0.7)",
        # )

    if all_vals:
        mn, mx = min(all_vals), max(all_vals)
        fig.add_shape(
            type="line", x0=mn, y0=mn, x1=mx, y1=mx,
            line=dict(dash="dash", color="black", width=1),
            row=row, col=col,
        )

    fig.update_xaxes(title_text="Actual (MWh)", showline=True, linecolor="black", mirror=True, row=row, col=col, tickfont=dict(size=16), )
    fig.update_yaxes(title_text="Predicted (MWh)", showline=True, linecolor="black", mirror=True, row=row, col=col, tickfont=dict(size=16), )



title_suffix = f"top {100 - EXTREME_PERCENTILE}% extremes" if EXCLUDE_EXTREMES else "all hours"
fig.update_layout(
    title=f"Actual vs Predicted by Zone: {title_suffix}",
    height=500 * n_rows,
    paper_bgcolor="white",
    plot_bgcolor="white",
    legend=dict(x=1.01, y=1),
    font=dict(size=18)
)
fig.update_annotations(font_size=18)
fig.show()


In [ ]:
fig.write_html("archive/model_comparison.html")

In [ ]:
def rmspe(actual, predicted):
    pct_err = (actual - predicted) / actual
    return np.sqrt((pct_err ** 2).mean()) * 100

def mape(actual, predicted):
    return ((actual - predicted).abs() / actual).mean() * 100

rows = []
for df, model in [(xgb_df, "XGBoost"), (gam_df, "GAM"), (lasso_df, "LASSO"), (mlp_df, "MLP"), (mlpp_df, "MLP 6H"), (mlptest_df, "MLP Test")]:
    if df.empty:
        continue
    if EXCLUDE_EXTREMES and (label != "LASSO"):
        z=z.nlargest(int(len(z) * 0.01), "actual")
    
    for zone in LRZ_IDS:
        z = df[df["zone"] == zone]
        if z.empty:
            continue
        rows.append({
            "model":     model,
            "zone":      zone,
            "RMSPE (%)": rmspe(z["actual"], z["predicted"]),
            "MAPE (%)": mape(z["actual"],  z["predicted"]),
        })

metrics_df = pd.DataFrame(rows).set_index(["model", "zone"]).round(2)
metrics_df

In [ ]:
# ── Percentage error histograms by zone, comparing models ────────────────────
n_cols = 3
n_rows = 2

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=LRZ_IDS,
    horizontal_spacing=0.08,
    vertical_spacing=0.14,
)

for i, zone in enumerate(LRZ_IDS):
    row = i // n_cols + 1
    col = i  % n_cols + 1

    for df, label, color in [
        (lasso_df, "LASSO",   "#F1D313"),
        (gam_df,   "GAM",     "#F44336"),
        (xgb_df,   "XGBoost", "#2196F3"),
        (mlp_df, "MLP",     "#54C82D"),
        (mlpp_df, "MLP 6H",  "#3F9724"),
        (mlptest_df, "MLP Test",  "#B076D6"),
    ]:
        if df.empty:
            continue
        z = df[df["zone"] == zone]
        if z.empty:
            continue

        pct_errors = (z["predicted"] - z["actual"]) / z["actual"] * 100

        fig.add_trace(
            go.Histogram(
                x=pct_errors,
                name=label,
                legendgroup=label,
                showlegend=(i == 0),
                marker_color=color,
                opacity=0.6,
                nbinsx=60,
                histnorm="probability density",
                hovertemplate="Error: %{x:.1f}%<br>Density: %{y:.4f}<extra>" + label + "</extra>",
            ),
            row=row, col=col,
        )

    fig.add_vline(x=0, line_width=1.5, line_dash="dash", line_color="black", row=row, col=col)

    fig.update_xaxes(title_text="% Error", showline=True, linecolor="black", mirror=True, row=row, col=col)
    fig.update_yaxes(title_text="Density",  showline=True, linecolor="black", mirror=True, row=row, col=col)

title_suffix = f"top {100 - EXTREME_PERCENTILE}% extremes" if EXCLUDE_EXTREMES else "all hours"
fig.update_layout(
    title=f"Prediction % Error Distribution by Zone — {title_suffix}",
    barmode="overlay",
    height=400 * n_rows,
    paper_bgcolor="white",
    plot_bgcolor="white",
    legend=dict(x=1.01, y=1),
)
fig.show()

In [ ]:
# ── Hourly time series: actual vs predicted for a date range and zone ────────
plot_zone       = "MISO-0027"   # zone to plot
plot_start      = "2022-06-20"  # start date (inclusive)
plot_end        = "2022-06-30"  # end date (inclusive)

fig = go.Figure()

for df, label, color, dash in [
    (xgb_df,   "XGBoost", "#2196F3", "solid"),
    (gam_df,   "GAM",     "#F44336", "dot"),
    (lasso_df, "LASSO",   "#E6C117", "dash"),
    (mlp_df, "MLP",     "#54C82D", "solid"),
]:
    if df.empty:
        continue
    z = df[df["zone"] == plot_zone].copy()
    if z.empty:
        continue
    z.index = pd.to_datetime(z.index)
    mask = (z.index >= plot_start) & (z.index <= plot_end)
    z = z.loc[mask]
    if z.empty:
        continue

    fig.add_trace(go.Scatter(
        x=z.index, y=z["predicted"],
        mode="lines",
        name=label,
        line=dict(color=color, dash=dash, width=1.5),
        hovertemplate="%{x|%Y-%m-%d %H:%M}<br>" + label + ": %{y:,.0f} MWh<extra></extra>",
    ))

# Actual load — use whichever df has data for this zone/period
for df in [xgb_df, gam_df, lasso_df]:
    if df.empty:
        continue
    z = df[df["zone"] == plot_zone].copy()
    if z.empty:
        continue
    z.index = pd.to_datetime(z.index)
    mask = (z.index >= plot_start) & (z.index <= plot_end)
    z = z.loc[mask]
    if z.empty:
        continue
    fig.add_trace(go.Scatter(
        x=z.index, y=z["actual"],
        mode="lines",
        name="Actual",
        line=dict(color="black", width=2),
        hovertemplate="%{x|%Y-%m-%d %H:%M}<br>Actual: %{y:,.0f} MWh<extra></extra>",
    ))
    break  # only need actual once

fig.update_layout(
    title=f"{plot_zone} — Hourly Load: {plot_start} to {plot_end}",
    xaxis_title="Date",
    yaxis_title="Load (MWh)",
    paper_bgcolor="white",
    plot_bgcolor="white",
    legend=dict(x=1.01, y=1),
    hovermode="x unified",
)
fig.update_xaxes(showline=True, linecolor="black", mirror=True)
fig.update_yaxes(showline=True, linecolor="black", mirror=True)
fig.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pygam import LinearGAM, s, f, te
import sys

zone = "MISO-0001"


# ── Constants (must match gam_worker.py) ──────────────────────────────────────
HIGH_LOAD_WEIGHT_PERCENTILE = 90
HIGH_LOAD_WEIGHT            = 1

CDD_BASE = 18.0  # °C (≈ 65°F) — standard US utility base temperature

# Compute training maxes + a buffer before building the formula
cdd_train = np.maximum(0.0, X_tr_fit["cdd"].values)
hdd_train = np.maximum(0.0, X_tr_fit["hdd"].values)

CDD_MAX = cdd_train.max() * 1.3   # 30% headroom above training max
HDD_MAX = hdd_train.max() * 1.3


GAM_COLS = [
    "cdd", "hdd",
    "cdd_lag1", "hdd_lag1",
    "cdd_lag24", "hdd_lag24",
    "hour", "month", "day_of_week", "year",
    "memorial_day", "july_4th", "labor_day", "christmas", "new_years",
]

# col indices: 0:cdd  1:hdd  2:cdd_lag1  3:hdd_lag1  4:cdd_lag24  5:hdd_lag24
#              6:hour  7:month  8:dow  9:year
#              10:memorial_day  11:july_4th  12:labor_day  13:christmas  14:new_years
GAM_FORMULA = (
    s(0,  n_splines=15)              # cdd
    + s(1,  n_splines=15)            # hdd
    + s(2,  n_splines=8)             # cdd_lag1
    + s(3,  n_splines=8)             # hdd_lag1
    + s(4,  n_splines=8)             # cdd_lag24
    + s(5,  n_splines=8)             # hdd_lag24
    + te(6, 7, n_splines=[24, 12])   # hour × month tensor product
    + f(8)                           # day_of_week
    + s(9,  n_splines=6)             # year
    + f(10)                          # memorial_day window
    + f(11)                          # july_4th window
    + f(12)                          # labor_day window
    + f(13)                          # christmas window
    + f(14)                          # new_years window
)


def build_features_raw(temp_series):
    t   = temp_series
    t_f = t * 9 / 5 + 32

    X = pd.DataFrame(index=t.index)
    X["cdd"]      = np.maximum(0.0, t - CDD_BASE)
    X["hdd"]      = np.maximum(0.0, CDD_BASE - t)
    X["cdd_lag1"]  = X["cdd"].shift(1)
    X["hdd_lag1"]  = X["hdd"].shift(1)
    X["cdd_lag24"] = X["cdd"].shift(24)
    X["hdd_lag24"] = X["hdd"].shift(24)

    daily_avg_f = t_f.resample("D").mean().reindex(t.index, method="ffill")
    bin_edges   = np.arange(
        np.floor(daily_avg_f.min() / 10) * 10,
        np.ceil(daily_avg_f.max()  / 10) * 10 + 10,
        10,
    )
    X["temp_bin"]    = pd.cut(daily_avg_f, bins=bin_edges, labels=False).astype(float)
    X["hour"]        = t.index.hour
    X["day_of_week"] = t.index.day_of_week
    X["weekend"]     = (t.index.day_of_week > 4).astype(int)
    X["month"]       = t.index.month
    X["year"]        = t.index.year

    date_index = t.index.normalize()
    if date_index.tz is not None:
        date_index = date_index.tz_localize(None)

    holiday_dates = get_holiday_dates(t.index.year.unique())
    for holiday, day_set in holiday_dates.items():
        col = np.zeros(len(t.index), dtype=int)
        for level, offset in enumerate(range(-HOLIDAY_WINDOW, HOLIDAY_WINDOW + 1), start=1):
            shifted = frozenset(d + pd.Timedelta(days=offset) for d in day_set)
            col[date_index.isin(shifted)] = level
        X[holiday] = col

    return X

# GAM_COLS = [
#     "temp", "temp_squared", "temp_lag1", "temp_lag24",
#     "hour", "month", "day_of_week", "year",
#     "memorial_day", "july_4th", "labor_day", "christmas", "new_years",
# ]

# # col indices: 0:temp  1:temp_sq  2:lag1  3:lag24  4:hour  5:month
# #              6:dow  7:year  8:memorial_day  9:july_4th  10:labor_day
# #              11:christmas  12:new_years
# GAM_FORMULA = (
#     s(0,  n_splines=20)              # temp
#     + s(1,  n_splines=10)            # temp_squared
#     + s(2,  n_splines=10)            # temp_lag1
#     + s(3,  n_splines=10)            # temp_lag24
#     + te(4, 5, n_splines=[24, 12])   # hour × month tensor product
#     + f(6)                           # day_of_week
#     + s(7,  n_splines=6)             # year
#     + f(8)                           # memorial_day window
#     + f(9)                           # july_4th window
#     + f(10)                          # labor_day window
#     + f(11)                          # christmas window
#     + f(12)                          # new_years window
# )



# # ── Feature engineering ──────────────────────────────────────────────────────
# def build_features_raw(temp_series):
#     """Build raw feature DataFrame for a single LRZ temperature series."""
#     t   = temp_series
#     t_f = t * 9 / 5 + 32

#     X = pd.DataFrame(index=t.index)
#     X["temp"]         = t
#     X["temp_squared"] = t ** 2
#     X["temp_lag1"]    = t.shift(1)
#     X["temp_lag24"]   = t.shift(24)

#     daily_avg_f = t_f.resample("D").mean().reindex(t.index, method="ffill")
#     bin_edges   = np.arange(
#         np.floor(daily_avg_f.min() / 10) * 10,
#         np.ceil(daily_avg_f.max()  / 10) * 10 + 10,
#         10,
#     )
#     X["temp_bin"]    = pd.cut(daily_avg_f, bins=bin_edges, labels=False).astype(float)
#     X["hour"]        = t.index.hour
#     X["day_of_week"] = t.index.day_of_week
#     X["weekend"]     = (t.index.day_of_week > 4).astype(int)
#     X["month"]       = t.index.month
#     X["year"]        = t.index.year

#     # Holiday window features
#     # Strip timezone so comparisons with tz-naive holiday dates work correctly.
#     date_index = t.index.normalize()
#     if date_index.tz is not None:
#         date_index = date_index.tz_localize(None)

#     holiday_dates = get_holiday_dates(t.index.year.unique())
#     # Encoding: 0 = not in window, 1 = -2 days, 2 = -1 day, 3 = day-of,
#     #           4 = +1 day, 5 = +2 days
#     for holiday, day_set in holiday_dates.items():
#         col = np.zeros(len(t.index), dtype=int)
#         for level, offset in enumerate(range(-HOLIDAY_WINDOW, HOLIDAY_WINDOW + 1), start=1):
#             shifted = frozenset(d + pd.Timedelta(days=offset) for d in day_set)
#             col[date_index.isin(shifted)] = level
#         X[holiday] = col

#     return X


# Build features for this zone (lags computed on full training series)
X_all = build_features_raw(temp[zone]).dropna()
y_all = load[zone].loc[X_all.index]
groups = pd.Series(X_all.index.year, index=X_all.index)
# Get all LOGO splits and select the requested fold
logo   = LeaveOneGroupOut()
splits = list(logo.split(X_all, y_all, groups))

train_idx, test_idx = splits[0]


X_tr = X_all[GAM_COLS].iloc[train_idx]
X_te = X_all[GAM_COLS].iloc[test_idx]
y_tr = y_all.iloc[train_idx]
y_te = y_all.iloc[test_idx]

# Optionally restrict training and evaluation to extremes
if EXCLUDE_EXTREMES:
    train_thresh = np.percentile(y_tr, EXTREME_PERCENTILE)
    X_tr_fit = X_tr[y_tr <= train_thresh]
    y_tr_fit = y_tr[y_tr <= train_thresh]

    test_thresh = np.percentile(y_te, EXTREME_PERCENTILE)
    extreme_mask = y_te > test_thresh
    X_te_eval = X_te #[extreme_mask]
    y_te_eval = y_te #[extreme_mask]
else:
    X_tr_fit, y_tr_fit = X_tr, y_tr
    X_te_eval, y_te_eval = X_te, y_te
    
# ── Sample weights: top HIGH_LOAD_WEIGHT_PERCENTILE% → HIGH_LOAD_WEIGHT ──────
w_thresh = np.percentile(y_tr_fit, HIGH_LOAD_WEIGHT_PERCENTILE)
weights  = np.where(y_tr_fit.values >= w_thresh, HIGH_LOAD_WEIGHT, 1.0)
print(f"Weighting {(y_tr_fit.values >= w_thresh).sum()} high-load points "
      f"(≥{w_thresh:.0f} MWh) at {HIGH_LOAD_WEIGHT}x")

# ── Fit ────────────────────────────────────────────────────────────────────────
gam = LinearGAM(GAM_FORMULA, max_iter=100)
gam.gridsearch(X_tr_fit[GAM_COLS].values, y_tr_fit.values, weights=weights, progress=True)

print(f"\nFitted GAM — lambda: {gam.lam}")
print(gam.summary())


def get_spline_knots(term, x_data):
    """Replicate pyGAM's internal knot placement: equally-spaced quantiles."""
    n_knots = term.n_splines - term.spline_order + 1
    return np.percentile(x_data, np.linspace(0, 100, n_knots))

# (term_index_in_formula, col_index_in_X, col_name, xlabel)
# TEMP_SPLINE_TERMS = [
#     (0, 0, "temp",         "temp (°C)"),
#     (1, 1, "temp_squared", "temp² (°C²)"),
#     (2, 2, "temp_lag1",    "temp_lag1 (°C)"),
#     (3, 3, "temp_lag24",   "temp_lag24 (°C)"),
# ]

TEMP_SPLINE_TERMS = [
    (0, 0, "cdd",         "cdd (°C)"),
    (1, 1, "hdd", "hdd (°C)"),
    (2, 2, "cdd_lag1",    "cdd_lag1 (°C)"),
    (3, 3, "hdd_lag1",   "hdd_lag1 (°C)"),
]


fig, axes = plt.subplots(len(TEMP_SPLINE_TERMS), 2, figsize=(14, 4 * len(TEMP_SPLINE_TERMS)))

for row, (term_idx, col_idx, col_name, xlabel) in enumerate(TEMP_SPLINE_TERMS):
    term   = gam.terms[term_idx]
    x_data = X_tr_fit[col_name].values

    knots          = get_spline_knots(term, x_data)
    internal_knots = knots[1:-1]

    # ── Left: data distribution + knot positions ──────────────────────────
    ax = axes[row, 0]
    ax.hist(x_data, bins=80, color="steelblue", alpha=0.55, density=True, label="training data")
    for k in internal_knots:
        ax.axvline(k, color="crimson", lw=1.0, alpha=0.8)
    ax.axvline(knots[0],  color="black", lw=1.2, ls="--", alpha=0.6, label="boundary knots")
    ax.axvline(knots[-1], color="black", lw=1.2, ls="--", alpha=0.6)
    ax.set_title(
        f"{col_name}  |  n_splines={term.n_splines}  →  "
        f"{len(internal_knots)} internal + 2 boundary knots",
        fontsize=10,
    )
    ax.set_xlabel(xlabel)
    ax.set_ylabel("density")
    ax.legend(fontsize=8)

    # ── Right: fitted partial dependence + knot tick marks ────────────────
    ax2 = axes[row, 1]
    XX   = gam.generate_X_grid(term=term_idx, n=500)
    pdep, conf = gam.partial_dependence(term=term_idx, X=XX, width=0.95)
    x_grid = XX[:, col_idx]
    ax2.plot(x_grid, pdep, color="navy", lw=2)
    ax2.fill_between(x_grid, conf[:, 0], conf[:, 1], alpha=0.18, color="navy", label="95% CI")
    for k in internal_knots:
        ax2.axvline(k, color="crimson", lw=0.8, alpha=0.6)
    ax2.axvline(knots[0],  color="black", lw=1.0, ls="--", alpha=0.5)
    ax2.axvline(knots[-1], color="black", lw=1.0, ls="--", alpha=0.5)
    ax2.set_title(f"Partial dependence: {col_name}", fontsize=10)
    ax2.set_xlabel(xlabel)
    ax2.set_ylabel("effect on load (MWh)")
    ax2.legend(fontsize=8)

plt.suptitle("Temperature spline knot positions (red = internal, dashed = boundary)", y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

# ── Print knot values ─────────────────────────────────────────────────────────
for term_idx, col_idx, col_name, _ in TEMP_SPLINE_TERMS:
    term  = gam.terms[term_idx]
    knots = get_spline_knots(term, X_tr_fit[col_name].values)
    print(f"\n{col_name}  (n_splines={term.n_splines}, spline_order={term.spline_order})")
    print(f"  boundary : [{knots[0]:.3f}, {knots[-1]:.3f}]")
    print(f"  internal : {np.round(knots[1:-1], 3).tolist()}")


In [ ]:
term_idx, col_idx = 0, 0
XX = gam.generate_X_grid(term=term_idx, n=500)

# Extend the grid manually beyond training range
x_min = X_tr_fit["temp"].min() - 5
x_max = X_tr_fit["temp"].max() + 5
XX_ext = np.tile(XX[0], (200, 1))
XX_ext[:, col_idx] = np.linspace(x_min, x_max, 200)

pdep_ext = gam.predict(XX_ext) - gam.predict(XX_ext)  # partial dep needs offset trick
# Simpler: use partial_dependence directly on the extended grid
pdep_ext, conf_ext = gam.partial_dependence(term=term_idx, X=XX_ext, width=0.95)

plt.plot(XX_ext[:, col_idx], pdep_ext, color="navy")
plt.fill_between(XX_ext[:, col_idx], conf_ext[:,0], conf_ext[:,1], alpha=0.2, color="navy")
plt.axvline(X_tr_fit["temp"].min(), color="red", ls="--", label="training boundary")
plt.axvline(X_tr_fit["temp"].max(), color="red", ls="--")
plt.xlabel("temp (°C)"); plt.ylabel("partial effect"); plt.legend(); plt.show()

## MLP Inference vs Historical Load — Ensemble 011, 2015

In [11]:
from pathlib import Path
from plotly.subplots import make_subplots
import plotly.graph_objects as go

INFER_DIR = Path("/glade/work/rbhandarkar/Inputs/Load Profiles/011")
HIST_DIR  = Path("/glade/u/home/rbhandarkar/Inputs/Load Regression/MISO historical loads")
ZONES     = ["MISO-0001", "MISO-0027", "MISO-0035", "MISO-0004", "MISO-0006", "MISO-8910"]
LOCAL_TZ  = "America/Indiana/Indianapolis"
YEAR      = 2025

title_map = {"MISO-0001" : "ND & MN", "MISO-0027" : "WI & MI", "MISO-0035": "IA & MO", "MISO-0004" : "IL", "MISO-0006" : "IN", "MISO-8910": "AR, MI, & LA"}

# ── Load inference CSVs → UTC ─────────────────────────────────────────────────
infer_utc = {}
for z in ZONES:
    df = pd.read_csv(INFER_DIR / f"{z}_load_{YEAR}.csv", index_col=0)
    df.index = pd.to_datetime(df.index, utc=True)
    infer_utc[z] = df["predicted_load_mwh"]

# ── Load historical MISO XLS → UTC → resample 6H in UTC ──────────────────────
hist_raw = pd.read_excel(
    HIST_DIR / f"{YEAR}1231_dfal_HIST.xls",
    index_col=0, header=5,
)
hist_raw = hist_raw[:-2].drop(columns=["MTLF (MWh)"])
hist_raw = hist_raw[~(hist_raw.index == "MarketDay")]
hist_raw.index = pd.to_datetime(hist_raw.index)
hist_raw.index = hist_raw.index + pd.to_timedelta(hist_raw["HourEnding"] - 1, unit="h")
# EST has no DST — localize as fixed UTC-5 then convert to UTC
hist_raw.index = hist_raw.index.tz_localize("Etc/GMT+5").tz_convert("UTC")

hist_wide = hist_raw.pivot_table(
    index=hist_raw.index, columns="LoadResource Zone", values="ActualLoad (MWh)"
)
# hist_wide["LRZ8_9_10"] = hist_wide["LRZ8_9_10"].fillna(0) + hist_wide["LRZ8_9"].fillna(0)
hist_wide = hist_wide.drop(columns=["MISO"])
hist_wide.columns = ZONES

# Resample in UTC so bin boundaries match inference timestamps
hist_6h_utc = hist_wide.resample("6h").mean()

# ── Align on common UTC index ─────────────────────────────────────────────────
common_utc = infer_utc[ZONES[0]].index.intersection(hist_6h_utc.index)
print(f"Common 6H timestamps: {len(common_utc)}")
print(f"  {common_utc[0]}  →  {common_utc[-1]}")

# Local-time index for x-axis display
common_local = common_utc.tz_convert(LOCAL_TZ)

# ── Plotly: 3×2 subplots, one per zone ───────────────────────────────────────
SMOOTH = 28   # 28 × 6h = 7 days

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=ZONES,
    shared_xaxes=True,
    vertical_spacing=0.08,
    horizontal_spacing=0.08,
)

for i, z in enumerate(ZONES):
    row, col = i // 2 + 1, i % 2 + 1

    pred   = infer_utc[z].loc[common_utc]
    actual = hist_6h_utc[z].loc[common_utc]

    pred_s   = pred
    actual_s = actual

    fig.add_trace(go.Scatter(
        x=common_local, y=actual_s.values,
        mode="lines", name="Historical",
        line=dict(color="steelblue", width=1.8),
        opacity=0.7,
        legendgroup="actual", showlegend=(i == 0),
        hovertemplate="%{x|%b %d}<br>Actual: %{y:,.0f} MWh<extra>Historical</extra>",
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=common_local, y=pred_s.values,
        mode="lines", name="MLP inference",
        line=dict(color="darkorange", width=1.8),
        opacity=0.7,
        legendgroup="pred", showlegend=(i == 0),
        hovertemplate="%{x|%b %d}<br>Predicted: %{y:,.0f} MWh<extra>MLP inference</extra>",
    ), row=row, col=col)

    # Annotate MAE/RMSE in subplot title
    fig.layout.annotations[i].text = (
        f"{title_map[z]}")

    fig.update_yaxes(title_text="Load (MWh)", row=row, col=col)
    if row == 3:
        fig.update_xaxes(title_text="Date (ET)", row=row, col=col)

fig.update_layout(
    title=f"MLP Inference vs Historical Load — {YEAR}  (ensemble 011)",
    height=800,
    paper_bgcolor="white",
    plot_bgcolor="white",
    hovermode="x unified",
    legend=dict(x=1.01, y=1),
)
fig.update_xaxes(showgrid=True, gridcolor="lightgrey")
fig.update_yaxes(showgrid=True, gridcolor="lightgrey")
fig.show()

Common 6H timestamps: 1453
  2025-01-02 00:00:00+00:00  →  2025-12-31 00:00:00+00:00


In [8]:
# ── Annual peak load day heatmap: month × hour, per zone ─────────────────────
# Peak-day analysis with hours in Central Time (DST-aware).
# Both month and hour are derived in local time (Indianapolis/Chicago).

SUMMER_MONTHS = {6, 7, 8}
WINTER_MONTHS = {12, 1, 2}
MONTH_LABELS  = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
HOURS         = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11,
                 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
HOUR_LABELS   = [f"{h:02d}:00" for h in HOURS]
CENTRAL_TZ    = "America/Chicago"   # DST-aware Central Time

# ── Load ALL historical years → per zone, UTC, 6H ────────────────────────────
all_hist_years = sorted(int(p.stem[:4]) for p in HIST_DIR.glob("*1231_dfal_HIST.xls"))
print(f"Historical years found: {all_hist_years}")

hist_zone_yearly = {z: {} for z in ZONES}
for yr in all_hist_years:
    df = pd.read_excel(HIST_DIR / f"{yr}1231_dfal_HIST.xls", index_col=0, header=5)
    df = df[:-2].drop(columns=["MTLF (MWh)"])
    df = df[~(df.index == "MarketDay")]
    df.index = pd.to_datetime(df.index)
    df.index = df.index + pd.to_timedelta(df["HourEnding"] - 1, unit="h")
    df.index = df.index.tz_localize("Etc/GMT+5").tz_convert("UTC")
    wide = df.pivot_table(index=df.index, columns="LoadResource Zone", values="ActualLoad (MWh)")
    if "LRZ8_9" in wide.columns:
        wide["LRZ8_9_10"] = wide.get("LRZ8_9_10", pd.Series(0, index=wide.index)).fillna(0) + wide["LRZ8_9"].fillna(0)
        wide = wide.drop(columns=["LRZ8_9"])
    wide = wide.drop(columns=[c for c in ["MISO"] if c in wide.columns])
    wide_6h = wide.resample("6h").mean()   # keep in UTC internally
    wide_6h.columns = ZONES
    for z in ZONES:
        hist_zone_yearly[z][yr] = wide_6h[z]

# ── Load ALL inference years → per zone, UTC ──────────────────────────────────
infer_years = sorted(int(p.stem.split("_")[-1]) for p in INFER_DIR.glob(f"{ZONES[0]}_load_*.csv"))
print(f"Inference years found: {infer_years}")

infer_zone_yearly = {z: {} for z in ZONES}
for yr in infer_years:
    for z in ZONES:
        df = pd.read_csv(INFER_DIR / f"{z}_load_{yr}.csv", index_col=0)
        df.index = pd.to_datetime(df.index, utc=True)   # keep in UTC internally
        infer_zone_yearly[z][yr] = df["predicted_load_mwh"]

# ── Find annual peak day per zone ─────────────────────────────────────────────
def peak_day_records_zone(zone_yearly):
    out = {}
    for z, yearly in zone_yearly.items():
        records = []
        for year, series in yearly.items():
            # Convert to Central Time for peak-day identification and hour labeling
            series_ct = series.copy()
            series_ct.index = series_ct.index.tz_convert(CENTRAL_TZ)

            # Peak day in Central Time dates
            peak_date_ct = series_ct.groupby(series_ct.index.date).max().idxmax()
            day_series_ct = series_ct[series_ct.index.date == peak_date_ct]

            peak_month = day_series_ct.idxmax().month   # CT month

            for ts, val in day_series_ct.items():
                records.append({
                    "year":  year,
                    "month": peak_month,   # Central Time month
                    "hour":  ts.hour,      # Central Time hour (DST-aware)
                    "load":  val,
                })
        out[z] = pd.DataFrame(records)
    return out

hist_recs  = peak_day_records_zone(hist_zone_yearly)
infer_recs = peak_day_records_zone(infer_zone_yearly)

# ── Summer / winter statistics ────────────────────────────────────────────────
def season_stats_zone(recs_by_zone, dataset_label):
    print(f"\n{'─'*60}\n{dataset_label}\n{'─'*60}")
    for z in ZONES:
        rec = recs_by_zone[z]
        if rec.empty:
            print(f"  {z}: no data"); continue
        peaks = rec.drop_duplicates("year")
        n = len(peaks)
        s = peaks["month"].isin(SUMMER_MONTHS).sum()
        w = peaks["month"].isin(WINTER_MONTHS).sum()
        detail = "  ".join(
            f"{int(r.year)}→{MONTH_LABELS[int(r.month)-1]}"
            for _, r in peaks.sort_values("year").iterrows()
        )
        print(f"  {title_map[z]:20s}  Summer {s}/{n}={s/n*100:.0f}%  "
              f"Winter {w}/{n}={w/n*100:.0f}%  |  {detail}")

season_stats_zone(hist_recs,  "Historical")
season_stats_zone(infer_recs, "Inference (ens. 011)")

def build_matrix(rec):
    mat = np.full((12, 24), np.nan)
    ann = [["" for _ in HOURS] for _ in range(12)]
    for mi, month in enumerate(range(1, 13)):
        for hi, hour in enumerate(HOURS):
            cell = rec[(rec["month"] == month) & (rec["hour"] == hour)]
            if len(cell):
                mat[mi, hi] = cell["load"].mean()
                ann[mi][hi] = ", ".join(str(int(y)) for y in sorted(cell["year"]))

    anchor_mask = ~np.isnan(mat)

    # ── Forward-fill within the row ───────────────────────────────────────
    mat_ff = pd.DataFrame(mat).ffill(axis=1).values
    has_future_anchor = np.zeros_like(anchor_mask)
    for hi in range(23):
        has_future_anchor[:, hi] = anchor_mask[:, hi+1:].any(axis=1)
    fill_mask = (~anchor_mask) & has_future_anchor
    mat_out = np.where(fill_mask, mat_ff, mat)

    # ── Wrap: fill trailing tail with the last anchor value, but only up  
    #    to (not including) the first anchor of the day ────────────────────
    for mi in range(12):
        row = mat_out[mi]
        anchor_cols = np.where(anchor_mask[mi])[0]
        if len(anchor_cols) == 0:
            continue
        first_anchor = anchor_cols[0]
        last_anchor  = anchor_cols[-1]
        if first_anchor == 0:
            continue   # no leading NaNs to wrap into
        last_val = mat[mi, last_anchor]
        # Fill from last_anchor+1 → end of row (trailing tail)
        row[last_anchor + 1:] = last_val
        # Fill from 0 → first_anchor-1 (leading head, i.e. wrap-around)
        row[:first_anchor] = last_val
        mat_out[mi] = row

    return mat_out, ann

# ── Plot helper ───────────────────────────────────────────────────────────────
def make_peak_figure(recs_by_zone, dataset_label, n_years):
    matrices = {z: build_matrix(recs_by_zone[z]) for z in ZONES}
    all_vals = np.concatenate([m.flatten() for m, _ in matrices.values()])
    vmin, vmax = float(np.nanmin(all_vals)), float(np.nanmax(all_vals))

    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=[title_map[z] for z in ZONES],
        vertical_spacing=0.12,
        horizontal_spacing=0.08,
    )

    for i, z in enumerate(ZONES):
        r, c = i // 3 + 1, i % 3 + 1
        mat, ann = matrices[z]

        hover = [[
            (f"<b>{title_map[z]}: {MONTH_LABELS[mi]}, {HOUR_LABELS[hi]} CT</b><br>"
             f"Avg load: {mat[mi,hi]:,.0f} MWh<br>Years: {ann[mi][hi]}")
            if not np.isnan(mat[mi, hi])
            else f"{MONTH_LABELS[mi]}, {HOUR_LABELS[hi]} CT: no peak"
            for hi in range(24)
        ] for mi in range(12)]

        fig.add_trace(go.Heatmap(
            z=mat,
            x=HOUR_LABELS,
            y=MONTH_LABELS,
            colorscale="YlOrRd",
            zmin=vmin, zmax=vmax,
            showscale=(i == len(ZONES) - 1),
            colorbar=dict(title="MWh", x=1.02, len=0.5, y=0.25),
            text=hover,
            hovertemplate="%{text}<extra></extra>",
        ), row=r, col=c)

        fig.update_yaxes(autorange="reversed", row=r, col=c)
        if c == 1:
            fig.update_yaxes(title_text="Month of peak day", row=r, col=c)
        if r == 2:
            fig.update_xaxes(title_text="Hour of peak day (Central Time)", row=r, col=c)

    fig.update_layout(
        title=f"Annual Peak Load Day: Month × Hour — {dataset_label} ({n_years} years, 6H, Central Time)",
        height=600, paper_bgcolor="white", plot_bgcolor="white",
    )
    return fig

make_peak_figure(hist_recs,  "MISO Actual",             len(all_hist_years)).show()
make_peak_figure(infer_recs, "MESACLIP ens. 011", len(infer_years)).show()

Historical years found: [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Inference years found: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034, 2035, 2036, 2037, 2038, 2039, 2040, 2041, 2042, 2043, 2044, 2045]

────────────────────────────────────────────────────────────
Historical
────────────────────────────────────────────────────────────
  ND & MN               Summer 13/13=100%  Winter 0/13=0%  |  2013→Aug  2014→Jul  2015→Aug  2016→Jul  2017→Jul  2018→Jun  2019→Jul  2020→Jul  2021→Jun  2022→Jun  2023→Jul  2024→Aug  2025→Jul
  WI & MI               Summer 12/13=92%  Winter 0/13=0%  |  2013→Jul  2014→Jul  2015→Jul  2016→Aug  2017→Jun  2018→Jun  2019→Jul  2020→Jul  2021→Aug  2022→Jun  2023→Sep  2024→Aug  2025→Jun
  IA & MO               Summer 13/13=100%  Winter 0/13=0%  |  2013→Aug  2014→Aug  2015→Jul  2016→Jul  2017→Jul  2018→Jul  2019→Jul  2020→Aug  2021→Aug  2022→Jul  2023→Aug  

In [9]:
# ── Annual peak load day heatmap: month × hour, per zone ─────────────────────
# Peak-day analysis in UTC so hours are always 0/6/12/18.
# Month is derived in local time (Indianapolis) for summer/winter classification.

SUMMER_MONTHS = {6, 7, 8}
WINTER_MONTHS = {12, 1, 2}
MONTH_LABELS  = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
HOURS         = [0, 6, 12, 18]
HOUR_LABELS   = ["00:00", "06:00", "12:00", "18:00"]

# ── Load ALL historical years → per zone, UTC, 6H ────────────────────────────
all_hist_years = sorted(int(p.stem[:4]) for p in HIST_DIR.glob("*1231_dfal_HIST.xls"))
print(f"Historical years found: {all_hist_years}")

hist_zone_yearly = {z: {} for z in ZONES}
for yr in all_hist_years:
    df = pd.read_excel(HIST_DIR / f"{yr}1231_dfal_HIST.xls", index_col=0, header=5)
    df = df[:-2].drop(columns=["MTLF (MWh)"])
    df = df[~(df.index == "MarketDay")]
    df.index = pd.to_datetime(df.index)
    df.index = df.index + pd.to_timedelta(df["HourEnding"] - 1, unit="h")
    df.index = df.index.tz_localize("Etc/GMT+5").tz_convert("UTC")
    wide = df.pivot_table(index=df.index, columns="LoadResource Zone", values="ActualLoad (MWh)")
    if "LRZ8_9" in wide.columns:
        wide["LRZ8_9_10"] = wide.get("LRZ8_9_10", pd.Series(0, index=wide.index)).fillna(0) + wide["LRZ8_9"].fillna(0)
        wide = wide.drop(columns=["LRZ8_9"])
    wide = wide.drop(columns=[c for c in ["MISO"] if c in wide.columns])
    wide_6h = wide.resample("6h").mean()   # keep in UTC
    wide_6h.columns = ZONES
    for z in ZONES:
        hist_zone_yearly[z][yr] = wide_6h[z]

# ── Load ALL inference years → per zone, UTC ──────────────────────────────────
infer_years = sorted(int(p.stem.split("_")[-1]) for p in INFER_DIR.glob(f"{ZONES[0]}_load_*.csv"))
print(f"Inference years found: {infer_years}")

infer_zone_yearly = {z: {} for z in ZONES}
for yr in infer_years:
    for z in ZONES:
        df = pd.read_csv(INFER_DIR / f"{z}_load_{yr}.csv", index_col=0)
        df.index = pd.to_datetime(df.index, utc=True)   # keep in UTC
        infer_zone_yearly[z][yr] = df["predicted_load_mwh"]

# ── Find annual peak day per zone ─────────────────────────────────────────────
def peak_day_records_zone(zone_yearly):
    out = {}
    for z, yearly in zone_yearly.items():
        records = []
        for year, series in yearly.items():
            # Peak day in UTC dates
            peak_date_utc = series.groupby(series.index.date).max().idxmax()
            day_series = series[series.index.date == peak_date_utc]
            # Local-time month of the peak timestamp (for season classification)
            peak_ts_local = day_series.idxmax().tz_convert(LOCAL_TZ)
            peak_month    = peak_ts_local.month
            for ts, val in day_series.items():
                records.append({
                    "year":  year,
                    "month": peak_month,   # local-time month
                    "hour":  ts.hour,      # UTC hour (always 0/6/12/18)
                    "load":  val,
                })
        out[z] = pd.DataFrame(records)
    return out

hist_recs  = peak_day_records_zone(hist_zone_yearly)
infer_recs = peak_day_records_zone(infer_zone_yearly)

# ── Summer / winter statistics ────────────────────────────────────────────────
def season_stats_zone(recs_by_zone, dataset_label):
    print(f"\n{'─'*60}\n{dataset_label}\n{'─'*60}")
    for z in ZONES:
        rec = recs_by_zone[z]
        if rec.empty:
            print(f"  {z}: no data"); continue
        peaks = rec.drop_duplicates("year")
        n = len(peaks)
        s = peaks["month"].isin(SUMMER_MONTHS).sum()
        w = peaks["month"].isin(WINTER_MONTHS).sum()
        detail = "  ".join(
            f"{int(r.year)}→{MONTH_LABELS[int(r.month)-1]}"
            for _, r in peaks.sort_values("year").iterrows()
        )
        print(f"  {title_map[z]:20s}  Summer {s}/{n}={s/n*100:.0f}%  "
              f"Winter {w}/{n}={w/n*100:.0f}%  |  {detail}")

season_stats_zone(hist_recs,  "Historical")
season_stats_zone(infer_recs, "Inference (ens. 011)")

# ── Build month × UTC-hour matrix ─────────────────────────────────────────────
def build_matrix(rec):
    mat = np.full((12, 4), np.nan)
    ann = [["" for _ in HOURS] for _ in range(12)]
    for mi, month in enumerate(range(1, 13)):
        for hi, hour in enumerate(HOURS):
            cell = rec[(rec["month"] == month) & (rec["hour"] == hour)]
            if len(cell):
                mat[mi, hi] = cell["load"].mean()
                ann[mi][hi] = ", ".join(str(int(y)) for y in sorted(cell["year"]))
    return mat, ann

# ── Plot helper ───────────────────────────────────────────────────────────────
def make_peak_figure(recs_by_zone, dataset_label, n_years):
    matrices = {z: build_matrix(recs_by_zone[z]) for z in ZONES}
    all_vals = np.concatenate([m.flatten() for m, _ in matrices.values()])
    vmin, vmax = float(np.nanmin(all_vals)), float(np.nanmax(all_vals))

    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=[title_map[z] for z in ZONES],
        vertical_spacing=0.16,
        horizontal_spacing=0.08,
    )

    for i, z in enumerate(ZONES):
        r, c = i // 3 + 1, i % 3 + 1
        mat, ann = matrices[z]

        hover = [[
            (f"<b>{title_map[z]}: {MONTH_LABELS[mi]}, {HOUR_LABELS[hi]} UTC</b><br>"
             f"Avg load: {mat[mi,hi]:,.0f} MWh<br>Years: {ann[mi][hi]}")
            if not np.isnan(mat[mi, hi])
            else f"{MONTH_LABELS[mi]}, {HOUR_LABELS[hi]} UTC: no peak"
            for hi in range(4)
        ] for mi in range(12)]

        fig.add_trace(go.Heatmap(
            z=mat,
            x=HOUR_LABELS,
            y=MONTH_LABELS,
            colorscale="YlOrRd",
            zmin=vmin, zmax=vmax,
            showscale=(i == len(ZONES) - 1),
            colorbar=dict(title="MWh", x=1.02, len=0.5, y=0.25),
            text=hover,
            hovertemplate="%{text}<extra></extra>",
        ), row=r, col=c)

        fig.update_yaxes(autorange="reversed", row=r, col=c)
        if c == 1:
            fig.update_yaxes(title_text="Month of peak day (local time)", row=r, col=c)
        if r == 2:
            fig.update_xaxes(title_text="Hour of peak day (UTC)", row=r, col=c)

    fig.update_layout(
        title=f"Annual Peak Load Day: Month × Hour — {dataset_label} ({n_years} years, 6H)",
        height=600, paper_bgcolor="white", plot_bgcolor="white",
    )
    return fig

make_peak_figure(hist_recs,  "MISO Actual",             len(all_hist_years)).show()
make_peak_figure(infer_recs, "MESACLIP ens. 011", len(infer_years)).show()

Historical years found: [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Inference years found: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034, 2035, 2036, 2037, 2038, 2039, 2040, 2041, 2042, 2043, 2044, 2045]

────────────────────────────────────────────────────────────
Historical
────────────────────────────────────────────────────────────
  ND & MN               Summer 13/13=100%  Winter 0/13=0%  |  2013→Aug  2014→Jul  2015→Aug  2016→Jul  2017→Jul  2018→Jun  2019→Jul  2020→Jul  2021→Jun  2022→Jun  2023→Jul  2024→Aug  2025→Jul
  WI & MI               Summer 12/13=92%  Winter 0/13=0%  |  2013→Jul  2014→Jul  2015→Jul  2016→Aug  2017→Jun  2018→Jun  2019→Jul  2020→Jul  2021→Aug  2022→Jun  2023→Sep  2024→Aug  2025→Jun
  IA & MO               Summer 13/13=100%  Winter 0/13=0%  |  2013→Aug  2014→Aug  2015→Jul  2016→Jul  2017→Jul  2018→Jul  2019→Jul  2020→Aug  2021→Aug  2022→Jul  2023→Aug  